<a href="https://colab.research.google.com/github/kwaka1208/colab/blob/main/%E7%94%BB%E5%83%8F%E3%81%AE%E7%89%B9%E5%BE%B4%E3%82%92%E3%82%AD%E3%83%BC%E3%83%AF%E3%83%BC%E3%83%89%E3%81%A7CSV%E3%81%AB%E5%87%BA%E5%8A%9B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import csv
import base64
from io import BytesIO
from PIL import Image
from openai import OpenAI
from google.colab import drive
from google.colab import userdata

# Google Drive をマウント
drive.mount('/content/drive')

# Google Drive をマウント
drive.mount('/content/drive')

# APIキーを環境変数から取得
APIKEY = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=APIKEY)

# 任意のGoogle Drive内フォルダを指定（例: '/content/drive/MyDrive/画像解析'）
input_folder = '/content/drive/MyDrive/画像解析'
output_csv = os.path.join(input_folder, 'output.csv')
num_keywords = 10 # 出力するキーワード数

# 画像をBase64に変換
def encode_image_to_base64(image_path):
    with Image.open(image_path) as img:
        buffered = BytesIO()
        img.save(buffered, format="PNG")
        return base64.b64encode(buffered.getvalue()).decode()

# ChatGPTに画像のキーワードを生成してもらう関数（日本語対応）
def generate_keywords(image_base64, num_keywords):
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "system",
                "content": "あなたは画像の特徴を日本語で表すキーワードを抽出するアシスタントです。"
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": f"この画像の特徴を表す日本語のキーワードを{num_keywords}個、箇条書きで挙げてください。"
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/png;base64,{image_base64}"
                        }
                    }
                ]
            }
        ],
        max_tokens=100
    )

    keywords_text = response.choices[0].message.content
    keywords = [k.strip("・-•\n ") for k in keywords_text.splitlines() if k.strip()]
    return keywords[:num_keywords]

# メイン処理
def process_images(folder_path, output_csv, num_keywords):
    rows = []
    for filename in os.listdir(folder_path):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            file_path = os.path.join(folder_path, filename)
            print(f"Processing: {filename}")
            try:
                base64_image = encode_image_to_base64(file_path)
                keywords = generate_keywords(base64_image, num_keywords)
                rows.append([filename] + keywords)
            except Exception as e:
                print(f"Failed to process {filename}: {e}")

    with open(output_csv, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerows(rows)
    print(f"CSV saved to {output_csv}")

# 実行
process_images(input_folder, output_csv, num_keywords)